# 프로젝트 Notebook 04. 공간 데이터와 Folium 지도

목표: 좌표를 검증하고 값의 크기를 과장하지 않는 지도를 만든 뒤 거리와 공간 집계의 한계를 설명한다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "data").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 실행하세요.")
print("저장소:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import folium
from math import radians, sin, cos, asin, sqrt
df = pd.read_csv(ROOT / "data/sample/location_features.csv")

## 1. 좌표 품질과 좌표계

WGS84 위도·경도를 가정한다. 목포 분석 범위를 벗어난 값과 결측을 검사한다.
Folium은 [위도, 경도] 순서이지만 GeoJSON은 흔히 [경도, 위도] 순서다.

In [ ]:
print(df[["위도", "경도"]].describe())
invalid = df.loc[
    ~df["위도"].between(33, 39) |
    ~df["경도"].between(124, 132) |
    df[["위도", "경도"]].isna().any(axis=1)
]
display(invalid)
assert invalid.empty

## 2. 원 면적에 값을 비례시키기

원의 면적은 반지름 제곱에 비례하므로 반지름에는 값의 제곱근을 사용한다.

In [ ]:
max_radius = 22
df["marker_radius"] = 4 + max_radius * np.sqrt(df["유동인구"] / df["유동인구"].max())
display(df[["행정동명", "유동인구", "marker_radius"]].head())

## 3. Folium 지도

In [ ]:
m = folium.Map(
    location=[df["위도"].mean(), df["경도"].mean()],
    zoom_start=13, tiles="CartoDB positron"
)
for _, row in df.iterrows():
    folium.CircleMarker(
        [row["위도"], row["경도"]],
        radius=row["marker_radius"],
        tooltip=f"{row['행정동명']} | 유동인구 {row['유동인구']:,}",
        color="#2563eb", fill=True, fill_opacity=.55,
    ).add_to(m)
m

## 4. Haversine 직선거리

$$a=\sin^2(\Delta\phi/2)+\cos\phi_1\cos\phi_2\sin^2(\Delta\lambda/2)$$

대표점 직선거리이며 실제 도보·차량 이동거리와 다르다.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = radians(lat1), radians(lat2)
    dp, dl = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dp/2)**2 + cos(p1)*cos(p2)*sin(dl/2)**2
    return 2 * 6371.0088 * asin(sqrt(a))

a, b = df.iloc[0], df.iloc[1]
distance = haversine_km(a["위도"], a["경도"], b["위도"], b["경도"])
print(a["행정동명"], b["행정동명"], round(distance, 3), "km")

## 5. 독립 연습

1. 표현 지표를 총인구, 카페수, 음식점수로 바꾸는 함수를 작성한다.
2. 색상은 적합도, 원 면적은 유동인구를 표현한다.
3. 선택한 행정동과 다른 모든 행정동 거리표를 작성한다.
4. 최근접 지역이 실제 이동시간에서도 최근접이라고 말할 수 없는 이유를 설명한다.
5. MAUP, 경계 효과와 생태학적 오류를 각각 현재 지도와 연결해 설명한다.

In [ ]:
# TODO: 지표 이름을 받아 지도를 반환하는 함수
def make_metric_map(data, metric):
    pass